In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

BASE_DIR = Path("../")

EXPORTS_DIR = BASE_DIR / "exports"

timeline_path = EXPORTS_DIR / "timeline_raw.csv"
ratings_path = EXPORTS_DIR / "ratings_raw.csv"
reviews_path = EXPORTS_DIR / "reviews_raw.csv"
ranks_path = EXPORTS_DIR / "ranks_raw.csv"

timeline_df = pd.read_csv(timeline_path)
ratings_df = pd.read_csv(ratings_path)
reviews_df = pd.read_csv(reviews_path)
ranks_df = pd.read_csv(ranks_path)

timeline_df["date"] = pd.to_datetime(timeline_df["date"])
ratings_df["date"] = pd.to_datetime(ratings_df["date"])
reviews_df["date"] = pd.to_datetime(reviews_df["date"])
ranks_df["date"] = pd.to_datetime(ranks_df["date"])

def remove_fully_null_columns(df):
    return df.dropna(axis=1, how="all")


timeline_df = remove_fully_null_columns(timeline_df)
ratings_df = remove_fully_null_columns(ratings_df)
reviews_df = remove_fully_null_columns(reviews_df)
ranks_df = remove_fully_null_columns(ranks_df)

ranks_df["grossing_rank_score"] = (
    1000 - ranks_df["store_product_rank_grossing"]
)

ratings_df["positive_rating_ratio"] = np.where(
    ratings_df["total_count_incremental"] > 0,
    ratings_df["five_star_incremental"] / ratings_df["total_count_incremental"],
    0
)

ratings_df["negative_rating_ratio"] = np.where(
    ratings_df["total_count_incremental"] > 0,
    ratings_df["one_star_incremental"] / ratings_df["total_count_incremental"],
    0
)

def classify_review_sentiment(rating):
    if rating >= 4:
        return "positive"
    elif rating == 3:
        return "neutral"
    else:
        return "negative"

reviews_df["review_sentiment"] = reviews_df["rating_value"].apply(classify_review_sentiment)

reviews_df["content"] = (
    reviews_df["content"]
    .astype(str)
    .str.strip()
)

reviews_df["title"] = (
    reviews_df["title"]
    .astype(str)
    .str.strip()
)

reviews_df["topics"] = (
    reviews_df["topics"]
    .astype(str)
    .str.lower()
    .str.strip()
)

reviews_df = reviews_df.drop_duplicates(
    subset=["review_id"]
)

for df in [
    timeline_df,
    ratings_df,
    reviews_df,
    ranks_df
]:
    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month
    df["week"] = df["date"].dt.isocalendar().week

datasets = {
    "timeline": timeline_df,
    "ratings": ratings_df,
    "reviews": reviews_df,
    "ranks": ranks_df
}

for name, df in datasets.items():
    print("\n" + "=" * 50)
    print(name.upper())
    print(df.head())
    print(df.shape)

timeline_df.to_csv(
    EXPORTS_DIR / "timeline_clean.csv",
    index=False
)

ratings_df.to_csv(
    EXPORTS_DIR / "ratings_clean.csv",
    index=False
)

reviews_df.to_csv(
    EXPORTS_DIR / "reviews_clean.csv",
    index=False
)

ranks_df.to_csv(
    EXPORTS_DIR / "ranks_clean.csv",
    index=False
)

print("\nData cleaning completed successfully")

print("\nFinal dataset shapes:")

print("Timeline:", timeline_df.shape)
print("Ratings:", ratings_df.shape)
print("Reviews:", reviews_df.shape)
print("Ranks:", ranks_df.shape)


TIMELINE
        date country_code device_code      app_id  product_id  \
0 2024-11-01           US     ios-all  1621328561  1621328561   
1 2024-11-14           US     ios-all  1621328561  1621328561   
2 2024-11-20           US     ios-all  1621328561  1621328561   
3 2024-11-27           US     ios-all  1621328561  1621328561   
4 2024-11-28           US     ios-all  1621328561  1621328561   

                   event_id event_type_name old_value new_value  \
0  67263c65a340ea2c2b02d4ae     new_version    1.33.5    1.34.1   
1  67376cbea340ea2c2b757678     new_version    1.34.1    1.35.0   
2  673e126ea340ea2c2b71fdda     new_version    1.35.0    1.35.1   
3  67477f2ca340ea2c2b5be68b     new_version    1.35.1    1.36.0   
4  674885a6a340ea2c2b19ef4e     new_version    1.36.0    1.36.6   

  compared_result                                      release_notes  \
0             NaN  So, what's new this time, Tycoons?\n- A new ki...   
1             NaN  This update brings you some thril